# Process

## Setup

- Imports
- Setup driver
- Open website
- Clear tracks
- Select layout
- Add all tracks and get their track names

## Scraping loop

For each rarity:
- Clear cars
- Open car search
- Open filters
- Add only this rarity
- Press done
- Repeatedly press show more
- Get total number of cars
- Make groups of 7
- For each group:
    - Add cars
    - Select tune 1 on all cars
    - Get all tune 1 times
    - For each car:
        - Open settings
        - Get tune 1 stats
        - Get car info
        - Select tune 2
        - Get tune 2 stats
    - Get all tune 2 times
    - For each remaining tune i:
        - For each car:
            - Open settings
            - Select tune i
            - Get tune i stats
        - Get all tune i times

In [3]:
scrape_new = True

# Setup

In [4]:
from typing import List, Tuple, Dict
import pyautogui
from selenium import webdriver
import pickle
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
import time
import pandas as pd
from selenium.common.exceptions import NoSuchElementException
from tqdm import tqdm

In [5]:

#    SSSS      CCCC   RRRRRR     AAA    PPPPPP  IIIIII  NN   NN    GGGGG
#   SS  SS    CC  CC  RR   RR   AA AA   PP   PP   II    NNN  NN   GG
#   SS       CC       RR   RR  AA   AA  PP   PP   II    NNNN NN  GG
#    SSSSS   CC       RR  RRR  AA   AA  PP   PP   II    NNNNNNN  GG  GGG
#        SS  CC       RRRRR    AAAAAAA  PPPPPP    II    NN NNNN  GG   GG
#   SS   SS   CC  CC  RR RRR   AA   AA  PP        II    NN  NNN   GG  GG
#    SSSSS     CCCC   RR  RRR  AA   AA  PP      IIIIII  NN   NN    GGGGG
#
#region

condition_map = {
    'Type_00': 'Dry',
    'Type_01': 'Wet',
    'Type_10': 'Dirt',
    'Type_20': 'Gravel',
    'Type_50': 'Sand',
    'Type_60': 'Snow',
    'Type_11': 'Dirt Wet',
    'Type_30': 'Ice',
    'Type_41': 'Aspht Dirt Wet',
    'Type_c0': 'Aspht Sand',
    'Type_40': 'Aspht Dirt',
    'Type_i0': 'Aspht',
    'Type_e0': 'Sand Dirt',
    'Type_f0': 'Aspht Grass Dirt',
    'Type_c1': 'Aspht Sand Wet',
    'Type_70': 'Grass',
    'Type_b0': 'Aspht Gravel',
    'Type_d0': 'Aspht Snow',
    'Type_71': 'Grass Wet',
    'Type_h1': 'Snow Dirt Wet',
    'Type_51': 'Sand Wet',
    'Type_m0': 'Aspht Dirt',
    'Type_k0': 'Dirt',
    'Type_g0': 'Ice Snow'
}

def click_element(driver: webdriver, element):
    last_exception = None
    for i in range(3):
        try:
            element.click()
            return
        except Exception as e:
            e_str = str(e)
            if 'BaseDialog_Back' in e_str:
                last_exception = e
                time.sleep(0.1)
            elif 'ElementClickInterceptedException' in e_str:
                last_exception = e
                time.sleep(1)
            else:
                raise
            
    raise last_exception


def press_menu_button(driver: webdriver) -> None:
    '''
    Presses the menu button on main page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
    
    Returns:
        None
    '''
    # Find menu button
    menu_button_xpath = '/html/body/div/div[2]/div[1]/div[1]/div[1]/div[1]/div[2]/div[1]/button[1]'
    # Click the menu button
    menu_el = driver.find_element(By.XPATH, menu_button_xpath)
    click_element(driver, menu_el)

def clear_cars(driver: webdriver) -> None:
    '''
    Presses the clear cars button on the menu.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
    
    Returns:
        None
    '''
    # Find clear cars button
    clear_cars_button_xpath = '/html/body/div/div[2]/div[8]/div[2]/div/div/div[1]/div[1]/div/button[2]'
    # Click the clear button
    clear_el = driver.find_element(By.XPATH, clear_cars_button_xpath)
    click_element(driver, clear_el)

def clear_trackset(driver: webdriver) -> None:
    '''
    Presses the clear trackset button on the menu.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
    
    Returns:
        None
    '''
    # Find clear trackset button
    clear_trackset_button_xpath = '/html/body/div/div[2]/div[8]/div[2]/div/div/div[1]/div[1]/div/button[1]'
    # Click the clear button
    clear_el = driver.find_element(By.XPATH, clear_trackset_button_xpath)
    click_element(driver, clear_el)

def click_off(y_coords: int) -> None:
    '''
    Clicks specific coordinates to close any pop-ups.

    Parameters:
        y_coords (int): Y coordinate to click off.

    Returns:
        None
    '''
    pyautogui.click(x=500, y=y_coords)

def initial_menu_setup(driver: webdriver, y_coords: int) -> None:
    '''
    Sets up the website for scraping. Clears cars and tracks, and selects correct layout.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        y_coords (int): Y-coordinate for clicking off the menu.
    
    Returns:
        None
    '''
    # Open menu
    press_menu_button(driver)

    # Clear cars
    clear_cars(driver)

    # Clear trackset
    clear_trackset(driver)

    # Set correct layout
    layout_button_xpath = '/html/body/div/div[2]/div[8]/div[2]/div/div/div[2]/div[1]/div[2]/button[3]'
    layout_el = driver.find_element(By.XPATH, layout_button_xpath)
    click_element(driver, layout_el)

    # Close menu
    click_off(y_coords)

def set_all_tracks(driver: webdriver, y_coords: int, test_mode = False) -> List:
    '''
    Selects every track and variant from the track selection, and gets their names.

    Parameters:
        driver (webdriver): The webdriver object
        y_coords (int): The y coordinate of the dropdown menu
        test_mode (bool): Whether to run in test mode or not (only adds first track)

    Returns:
        List: A list containing all track and variant combinations
    '''
    # Initialise trackset
    trackset = []

    # Open track selection
    add_tracks_button_xpath = '/html/body/div/div[2]/div[1]/div[1]/div[2]/div/div[2]/button'
    add_tracks_button = driver.find_element(By.XPATH, add_tracks_button_xpath)
    click_element(driver, add_tracks_button)
    time.sleep(0.1)

    # Get location of all track divs
    tracks_box_xpath = '/html/body/div/div[5]/div[2]/div/div/div'
    tracks_box = driver.find_element(By.XPATH, tracks_box_xpath)
    indiv_track_divs = tracks_box.find_elements(By.XPATH, './div')

    # Remove searchbar div
    indiv_track_divs = [div for div in indiv_track_divs if div.get_attribute('class') != 'Track_SearchBox']

    # Count number of tracks
    num_tracks = len(indiv_track_divs)

    # Create progress bar
    pbar = tqdm(total=num_tracks, desc='Adding tracks')


    # Loop through all main divs
    for track_div in indiv_track_divs:
        # Get track name
        track_name = track_div.find_element(By.XPATH, './div[1]/div').text

        # Check if track has other conditions (rolling start, bumps)
        # Try to find svg element, indicating other conditions
        try:
            i_el_class = track_div.find_element(By.TAG_NAME, 'i').get_attribute('class')
            if 'clearance' in i_el_class:
                track_name += ' (C)'
            if 'roll' in i_el_class:
                track_name += ' (R)'
        # If no i element, there are no other conditions
        except NoSuchElementException:
            pass

        # Get all buttons for track (now divs)
        conditions_container = track_div.find_element(By.XPATH, './div[2]/div')
        button_divs = conditions_container.find_elements(By.XPATH, './div')

        # Iterate through all buttons
        for b_div in button_divs:
            # Click button
            click_element(driver, b_div)

            # Get track condition type
            b_div_class = b_div.get_attribute('class')
            condition = b_div_class.split(' ')[2]

            # Map condition
            condition_str = condition_map[condition]

            # Append track with condition to trackset
            trackset.append((track_name, condition_str))

        # Update progress bar
        pbar.update(1)

        # If test mode, only add first track (all variants)
        if test_mode:
            break

    # Close track selection
    click_off(y_coords)

    # Close progress bar
    pbar.close()

    # Return the trackset
    return trackset


def open_car_search(driver: webdriver) -> None:
    '''
    Opens the car search menu from main page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None
    '''
    # Find car search button
    search_button_xpath = '/html/body/div/div[2]/div[1]/div[2]/div[2]/div[1]/div/button'
    # Click the search button
    try:
        search_button = driver.find_element(By.XPATH, search_button_xpath)
        click_element(driver, search_button)
    except:
        click_off(y_coords)
        driver.execute_script('window.scrollBy(0, 1)')
        driver.execute_script('window.scrollBy(0, -1);')
        search_button = driver.find_element(By.XPATH, search_button_xpath)
        click_element(driver, search_button)


def press_filters(driver: webdriver) -> None:
    '''
    Presses the filters button on the car search page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None
    '''
    # Find filters button
    filters_button_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[1]/button'
    # Click the filters button
    filter_el = driver.find_element(By.XPATH, filters_button_xpath)
    click_element(driver, filter_el)

def press_clear_filters(driver: webdriver) -> None:
    '''
    Presses clear filters button on filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None
    '''
    # Find clear filters button
    clear_filters_button_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[1]/div/button'
    # Click the clear filters button
    clear_el = driver.find_element(By.XPATH, clear_filters_button_xpath)
    click_element(driver, clear_el)

def select_rarity(driver: webdriver, rarity: str) -> None:
    '''
    Selects the rarity filter on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        rarity (str): Rarity to select.

    Returns:    
        None
    '''
    # Find rarity buttons
    rarity_button_container_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[2]'
    rarity_button_container = driver.find_element(By.XPATH, rarity_button_container_xpath)
    rarity_buttons = rarity_button_container.find_elements(By.TAG_NAME, 'button')

    # Go through buttons until we find the right one
    for button in rarity_buttons:
        # Get rarity
        button_rarity = button.find_element(By.TAG_NAME, 'span').text
        # If rarity matches, click button
        if button_rarity == rarity:
            click_element(driver, button)
            break

def press_done(driver: webdriver) -> None:
    '''
    Presses done on filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None
    '''
    # Find done button
    done_button_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[1]/button'
    # Click done button
    done_el = driver.find_element(By.XPATH, done_button_xpath)
    click_element(driver, done_el)

def get_car_count(driver: webdriver) -> int:
    '''
    Gets the number of cars for current filter selection, from car search page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        int: Number of cars for current filter selection.
    '''
    # Find car count element
    car_count_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[1]/button/div[2]'
    car_count_element = driver.find_element(By.XPATH, car_count_xpath)
    # Get car count text
    car_count_text = car_count_element.text

    return int(car_count_text[1:-1])

def split_into_sevens(n: int) -> List[Tuple[int, int]]:
    '''
    Splits a number into groups of 7 and 8.
    The function tries to maximise the number of groups of 7 and minimise the number of groups of 8.

    Parameters:
        n (int): The number to split.

    Returns:
        List[Tuple[int, int]]: A list of tuples, each containing the start and end of a group.
    '''
    # Tries to split n into groups of 7 with 8s if necessary
    # n = i*7 + j*8 where we try to maximise i and minimise j

    # Initialise list to hold groups
    groups = []

    # If divisible by 7, easy
    if n % 7 == 0:
        for i in range(0, n, 7):
            groups.append((i, i+7))
    
    # If not divisible by 7...
    else:
        # Initialise success bool
        solved = False
        # Check the max number of groups of 8
        max_eights = n // 8
        # Increment through number of groups of eight
        for num_eights in range(1, max_eights+1):
            # Check if using this many eights results in success
            if (n - num_eights*8) % 7 == 0:
                # Success
                # Make groups
                for i in range(0, n-num_eights*8, 7):
                    groups.append((i, i+7))
                for j in range(n-num_eights*8, n, 8):
                    groups.append((j, j+8))
                solved = True
                break
        # If we finish loop and have no success, add groups of 7 until less than 7 then add rest
        if not solved:
            for i in range(0, n, 7):
                groups.append((i, min(i+7, n)))

    return groups

def repeatedly_press_show_more(driver: webdriver) -> None:
    '''
    Presses the "show more" button repeatedly until no more cars to show in the car search page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None 
    '''
    # Initialise show more bool
    show_more = True
    # Begin loop
    while show_more:
        # Try to find show more button
        try:
            show_more_button_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div[2]/button'
            # Click show more button
            more_el = driver.find_element(By.XPATH, show_more_button_xpath)
            click_element(driver, more_el)
            time.sleep(0.1)
        except NoSuchElementException:
            # If no such element, break loop
            show_more = False

def add_car_group(driver: webdriver, group: Tuple[int, int]) -> None:
    '''
    Adds a group of cars to the comparison from the car search.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        group (Tuple[int, int]): The start and end of the group to add.

    Returns:
        None
    '''
    # Identify all cars in search results
    search_results_div_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]'
    search_results_div = driver.find_element(By.XPATH, search_results_div_xpath)
    car_buttons = search_results_div.find_elements(By.XPATH, './button')

    # Filter to buttons in the group
    group_buttons = car_buttons[group[0]:group[1]]

    # Click each button in the group
    for button in group_buttons:

        button_clicked = False
        while not button_clicked:
            # Try to click button
            try:
                click_element(driver, button)
                button_clicked = True
            except Exception as e:
                # In unsuccessful, check error
                # If error is an 'obsures' error, scroll down and try again, otherwise raise error
                if 'obscures' in str(e):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(0.1)
                else:
                    raise e

def select_first_tunes(driver: webdriver, y_coords: int) -> None:
    '''
    Selects the first tune from the car comparison page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.

    Returns:
        None
    '''
    # Find car rows
    car_list_xpath = '/html/body/div/div[2]/div[1]/div[2]/div[2]'
    car_list = driver.find_element(By.XPATH, car_list_xpath)
    car_rows = car_list.find_elements(By.XPATH, './div')[:-1]   # Last div is add car button
    
    # Click off


    # Iterate through car rows
    for car_row in car_rows:
        # Find tune button
        tune_button = car_row.find_element(By.XPATH, './div[2]/div/div[1]/div/div/button[1]')
        # Click tune button
        try:
            click_element(driver, tune_button)
        except:
            click_off(y_coords)
            driver.execute_script('window.scrollBy(0, 1)')
            driver.execute_script('window.scrollBy(0, -1);')
            tune_button = car_row.find_element(By.XPATH, './div[2]/div/div[1]/div/div/button[1]')
            click_element(driver, tune_button)
    
    # Move mouse to near top left corner
    pyautogui.moveTo(100, 100)
    # Wiggle mouse
    pyautogui.moveRel(200, 200, duration=0.1)

def scrape_times(driver: webdriver, trackset: List[str], tune: Tuple [int, int, int]) -> list:
    '''
    Gets the times and stats for the current tune of cars in the comparison page.

    Parameters:
        driver (webdriver): The webdriver object
        trackset (List[str]): The list of tracks to get times for
        tune (Tuple): The tuple of the tune to get times for (engine, weight, chassis)

    Returns:
        list: A list of dictionaries, each containing the times and stats for a car
    '''
    # Initialise a list to store car dicts
    car_dict_list = []

    # Get html code with all times in
    main_car_list_div = driver.find_element(By.XPATH, '/html/body/div/div[2]/div[1]/div[2]/div[2]')
    main_car_list_html = main_car_list_div.get_attribute('innerHTML')

    # Split into individual rows
    car_rows_html_list = main_car_list_html.split('<div id="Car_Layout')[1:]

    # Iterate through each car row
    for car_row_html in car_rows_html_list:
        # Get car info
        rq = car_row_html.split('BaseCard_Header2Right2">')[1].split('</div>')[0].strip()
        make = car_row_html.split('</b>')[1].split('</div>')[0].strip()
        model = car_row_html.split('_Header2Bottom">')[1].split('</div>')[0].strip()
        make_model = car_row_html.split('class="Car_HeaderName')[1].split('>')[1].split('</div')[0].strip()
        year = car_row_html.split('Car_HeaderBlockYear"')[1].split('>')[1].split('</div')[0].strip()
        stats = car_row_html.split('Car_HeaderBackDropRight')[1].split('Car_HeaderStatValue">')[1:]
        stats = [stat.split('<')[0] for stat in stats]

        # Split into individual tracks
        tracks_html_list = car_row_html.split('class="Row_Content"')[1:]

        # Extract times for each track
        track_times_list = [track_html.split('<!---->')[0].split('>')[1] for track_html in tracks_html_list]

        # Create a dict for this car
        car_dict = dict(zip(trackset, track_times_list))

        # Add car info and stats to car dict
        car_dict['rq'] = rq
        car_dict['make'] = make
        car_dict['model'] = model
        car_dict['make_model'] = make_model
        car_dict['year'] = year
        car_dict['top_speed'] = stats[0]
        car_dict['zero_sixty'] = stats[1]
        car_dict['handling'] = stats[2]
        car_dict['engine_up'] = tune[0]
        car_dict['weight_up'] = tune[1]
        car_dict['chassis_up'] = tune[2]

        # Add car dict to list
        car_dict_list.append(car_dict)
        
    return car_dict_list

def count_car_divs(driver: webdriver) -> List:
    '''
    Identifies all car divs

    Parameters:
        driver (webdriver): The webdriver object

    Returns:
        List: A list of car divs
    '''
    # Identify all car divs
    car_container = driver.find_element(By.CSS_SELECTOR, '.Main_CarList')
    car_divs = car_container.find_elements(By.XPATH, './div')
    # Remove divs with no id (add car button)
    car_divs = [div for div in car_divs if div.get_attribute('id')]

    return len(car_divs)

def change_tune(driver: webdriver, y_coords: int, tune: int, get_info: bool) -> List[Dict]:
    '''
    Changes the tune of all cars in the comparison page and gets their info if required.

    Parameters:
        driver (webdriver): The webdriver object
        y_coords (int): The y coordinate of the dropdown menu
        tune (int): The tune to select (1-4)
        get_info (bool): Whether to get car info or not

    Returns:
        List[Dict]: A list of dictionaries containing car info if get_info is True, otherwise None
    '''
    # If getting car info, initialise list to store car dicts
    if get_info:
        car_dict_list = []

    # Count number of cars
    num_cars = count_car_divs(driver)

    # Iterate through all cars
    for i in range(num_cars):
        # Identify settings button of car
        settings_button_xpath = f'/html/body/div/div[2]/div[1]/div[2]/div[2]/div[{i+1}]/div[2]/div/div[1]/div/div[2]/button'
        # Identify settings button
        settings_button = driver.find_element(By.XPATH, settings_button_xpath)

        try:
            click_element(driver, settings_button)
        except Exception as e:
            click_off(y_coords)
            driver.execute_script('window.scrollBy(0, 1)')
            driver.execute_script('window.scrollBy(0, -1);')
            settings_button = driver.find_element(By.XPATH, settings_button_xpath)
            click_element(driver, settings_button)

        # Find tune button
        tune_button_xpath = f'/html/body/div/div[2]/div[4]/div[2]/div/div/div/div[2]/button[{tune}]'
        # Click tune button
        try:
            tune_button = driver.find_element(By.XPATH, tune_button_xpath)
            click_element(driver, tune_button)
        except:
            click_off(y_coords)
            driver.execute_script('window.scrollBy(0, 1)')
            driver.execute_script('window.scrollBy(0, -1);')
            tune_button = driver.find_element(By.XPATH, tune_button_xpath)
            click_element(driver, tune_button)

        # If getting car info, get tyre, drive, ...
        if get_info:
            # Get html of useful info
            car_html_xpath = '/html/body/div/div[2]/div[4]/div[2]/div/div/div[2]'
            car_html = driver.find_element(By.XPATH, car_html_xpath).get_attribute('innerHTML')

            car_card = car_html.split('Row_DialogCardCard')[1]
            # On Card
            rid = car_card.split('.jpg')[0].split('/')[-1].strip()
            rq = car_card.split('class="Car_HeaderRQValue')[1].split('>')[1].split('</div')[0].strip()
            make_model = car_card.split('class="Car_HeaderName')[1].split('>')[1].split('</div')[0].strip()
            year = car_card.split('Car_HeaderBlockYear')[1].split('>')[1].split('</div')[0].strip()
            country = car_card.split('Car_HeaderBlockCountry')[1].split('>')[1].split('</div')[0].strip()
            tyres = car_card.split('Car_HeaderBlockTiresValue')[1].split('>')[1].split('</span')[0].strip()
            drive = car_card.split('Car_HeaderStatLabelDrive')[1].split('>')[1].split('</div')[0].strip()
            prize = 'Yes' if 'Car_HeaderBlockPrize' in car_card else 'No'

            # Tags
            tags_div = car_html.split('Row_DialogCardTags')[1].split('Row_DialogCardDual')[0]
            tags = [tag.split('>')[-1].strip() for tag in tags_div.split('</div>')[:-2]]
            tags_str = '/'.join(tags)

            # Other Stats
            other_stats_html = car_html.split('Row_DialogCardBottom')[1]
            other_stats_htmls = other_stats_html.split('Row_DialogCardStatValue')[1:]
            abs_flag = other_stats_htmls[0].split('>')[1].split('</div')[0].strip()
            tcs_flag = other_stats_htmls[1].split('>')[1].split('</div')[0].strip()
            clearance = other_stats_htmls[2].split('>')[1].split('</div')[0].strip()
            try:
                mra = other_stats_htmls[3].split('span')[1].split('>')[1].split('<')[0].strip()
            except IndexError:
                mra = ''
            weight = other_stats_htmls[4].split('>')[1].split('</div')[0].strip()
            fuel = other_stats_htmls[5].split('>')[1].split('</div')[0].strip()
            seats = other_stats_htmls[6].split('>')[1].split('</div')[0].strip()
            engine_pos = other_stats_htmls[7].split('>')[1].split('</div')[0].strip()
            body = other_stats_htmls[8].split('</div')[0].split('>')[-1].strip().replace(',&nbsp;', '/')
            brakes = other_stats_htmls[9].split('>')[1].split('<')[0].strip()

            # Create car dict
            car_dict = {
                'rid': rid,
                'rq': rq,
                'make_model': make_model,
                'year': year,
                'country': country,
                'tyres': tyres,
                'drive': drive,
                'prize': prize,
                'tags': tags_str,
                'abs': abs_flag,
                'tcs': tcs_flag,
                'clearance': clearance,
                'mra': mra,
                'weight': weight,
                'fuel': fuel,
                'seats': seats,
                'engine_pos': engine_pos,
                'body': body,
                'brakes': brakes
            }

            # Add car dict to list
            car_dict_list.append(car_dict)

        # Click off settings menu
        click_off(y_coords)
        
    # If getting car info, return list of car dicts
    if get_info:
        return car_dict_list
    else:
        return None

def select_tags(driver: webdriver, tags: List[str], print_add: bool=False) -> None:
    '''
    Selects the tags on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        tags (List[str]): List of tags to select.

    Returns:
        None
    '''
    # Find tag groups
    tag_group_1_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[15]'
    tag_group_1 = driver.find_element(By.XPATH, tag_group_1_xpath)
    tag_group_2_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[16]'
    tag_group_2 = driver.find_element(By.XPATH, tag_group_2_xpath)
    tag_group_3_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[17]'
    tag_group_3 = driver.find_element(By.XPATH, tag_group_3_xpath)
    tag_group_4_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[18]'
    tag_group_4 = driver.find_element(By.XPATH, tag_group_4_xpath)
    tag_groups = [tag_group_1, tag_group_2, tag_group_3, tag_group_4]

    # Iterate through tag
    if print_add:
        print('Adding tags')
    for tag_group in tag_groups:
        # Get all buttons in group
        tag_buttons = tag_group.find_elements(By.XPATH, './/button')
        # Iterate through buttons
        for button in tag_buttons:
            # Get tag name
            button_tag = button.find_element(By.TAG_NAME, 'span').text
            # If tag is in list, click button
            if button_tag in tags:
                click_element(driver, button)
                print(' ', button_tag)

def select_makes(driver: webdriver, makes: List[str]) -> None:
    '''
    Selects the makes on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        makes (List[str]): List of makes to select.

    Returns:
        None
    '''
    # Find make group
    make_group_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[20]'
    make_group = driver.find_element(By.XPATH, make_group_xpath)

    # Get all buttons in group
    make_buttons = make_group.find_elements(By.XPATH, './/button')
    # Iterate through buttons
    print('Adding makes')
    for button in make_buttons:
        # Get make name
        button_make = button.find_element(By.TAG_NAME, 'span').text
        # If make is in list, click button
        if button_make in makes:
            click_element(driver, button)
            print(' ', button_make)

def select_bodies(driver: webdriver, bodies: List[str], print_add: bool=False) -> None:
    '''
    Selects the bodies on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        bodies (List[str]): List of bodies to select.

    Returns:
        None
    '''
    # Find body group
    body_group_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[11]'
    body_group = driver.find_element(By.XPATH, body_group_xpath)

    # Get all buttons in group
    body_buttons = body_group.find_elements(By.XPATH, './/button')
    # Iterate through buttons
    if print_add:
        print('Adding bodies')
    for button in body_buttons:
        # Get body name
        button_body = button.find_element(By.TAG_NAME, 'span').text
        # If body is in list, click button
        if button_body in bodies:
            click_element(driver, button)
            print(' ', button_body)

def select_tyres(driver: webdriver, tyres: List[str], print_add: bool=False) -> None:
    '''
    Selects the tyres on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        tyres (List[str]): List of tyres to select.

    Returns:
        None
    '''
    # Find tyre group
    tyre_group_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[5]/div[1]'
    tyre_group = driver.find_element(By.XPATH, tyre_group_xpath)

    # Get all tyres in group
    tyre_buttons = tyre_group.find_elements(By.XPATH, './/button')
    # Iterate through buttons
    if print_add:
        print('Adding tyres')
    for button in tyre_buttons:
        # Get tyre name
        button_tyre = button.find_element(By.TAG_NAME, 'span').text
        # If tyre is in list, click button
        if button_tyre in tyres:
            click_element(driver, button)
            print(' ', button_tyre)

def select_countries(driver: webdriver, countries: List[str], print_add: bool=False) -> None:
    '''
    Selects the countries on the filters page.

    Parameters:
        driver (webdriver): Selenium webdriver instance.
        countries (List[str]): List of countries to select.

    Returns:
        None
    '''
    # Find tyre group
    country_group_xpath = '/html/body/div/div[2]/div[3]/div[2]/div/div/div[2]/div/div[9]'
    country_group = driver.find_element(By.XPATH, country_group_xpath)

    # Get all countries in group
    country_buttons = country_group.find_elements(By.XPATH, './/button')
    # Iterate through buttons
    if print_add:
        print_add: bool=False
    print('Adding countries')
    for button in country_buttons:
        # Get country name
        button_src = button.find_element(By.TAG_NAME, 'span').get_attribute('innerHTML')
        button_country = button_src.split('flags/')[1][:2]
        # If country is in list, click button
        if button_country in countries:
            click_element(driver, button)
            print(' ', button_country)

def scrape_main(driver: webdriver, trackset: List[str], scraping_progress: dict, y_coords: int, scrape_rarities: List[str]=[], scrape_tags: List[str]=[], scrape_makes: List[str]=[], scrape_bodies: List[str]=[], scrape_tyres: List[str]=[], scrape_countries: List[str]=[]) -> dict:
    # Define rarities
    rarities = ['S', 'A', 'B', 'C', 'D', 'E', 'F']

    # Extract scraping progress from dict
    car_times_and_stats_dicts = scraping_progress['car_times_and_stats_dicts']
    car_info_dicts = scraping_progress['car_info_dicts']
    scraped_groups = scraping_progress['scraped_groups']
    completed_rarities = scraping_progress['completed_rarities']

    first_scrape_rarity = True

    # Iterate through rarities
    for rarity in rarities:

        # If scrape_rarities provided, check if rarity is in it
        if scrape_rarities and rarity not in scrape_rarities:
            print(f'Skipping rarity {rarity}.')
            continue

        # If rarity already scraped, skip to next rarity
        if rarity in completed_rarities:
            print(f'Rarity {rarity} already scraped.')
            continue

        print_add = True if first_scrape_rarity else False

        print(f'Scraping rarity {rarity}.', end=' ')
        if len(scraped_groups[rarity]) > 0:
            print(f'{len(scraped_groups[rarity])} groups already scraped.')
        else:
            print('No groups already scraped.')

        # Open car search
        open_car_search(driver)
        
        # Open filter selection
        press_filters(driver)
        
        # Clear filters
        press_clear_filters(driver)

        # If tags provided, select them

        if scrape_tags:
            select_tags(driver, scrape_tags, print_add)
        
        # If makes provided, select them
        if scrape_makes:
            select_makes(driver, scrape_makes, print_add)
        
        # If bodies provided, select them
        if scrape_bodies:
            select_bodies(driver, scrape_bodies, print_add)

        # If tyres provided, select them
        if scrape_tyres:
            select_tyres(driver, scrape_tyres, print_add)
        
        if scrape_countries:
            select_countries(driver, scrape_countries, print_add)
                
        # Select rarity
        select_rarity(driver, rarity)

        time.sleep(1)

        
        # Press done
        press_done(driver)
        time.sleep(0.5)

        # Check number of cars
        car_count = get_car_count(driver)
        
        # Split into groups of 7, with 8s if needed
        groups = split_into_sevens(car_count)
        
        # Press show more until all cars showing in search
        repeatedly_press_show_more(driver)
        time.sleep(0.5)

        # Initialise pbar
        pbar = tqdm(total=len(groups), desc=f'Adding {rarity} cars', initial=len(scraped_groups[rarity]))
        
        # For each group:
        for group in groups:

            # If group already scraped, skip to next group
            if group in scraped_groups[rarity]:
                continue
        
            # Add cars
            add_car_group(driver, group)
        
            # Click off search
            click_off(y_coords)
            time.sleep(0.1)

            # Select tune 1 on all cars
            select_first_tunes(driver, y_coords)
        
            # Get times and stats for tune 1
            car_times_and_stats_dicts += scrape_times(driver, trackset, (3,3,2))
        
            # Select tune 2 on all cars, in the process get info of car
            car_info_dicts += change_tune(driver, y_coords, 2, True)
        
            # Get times and stats for tune 2
            car_times_and_stats_dicts += scrape_times(driver, trackset, (3,2,3))
        
            # Select tune 3
            change_tune(driver, y_coords, 3, False)
        
            # Get times and stats for tune 3
            car_times_and_stats_dicts += scrape_times(driver, trackset, (2,3,3))
        
            # If A/S:
            if rarity in ['A', 'S']:

                # Select tune 4
                change_tune(driver, y_coords, 4, False)
        
                # Get times and stats for tune 4
                car_times_and_stats_dicts += scrape_times(driver, trackset, (1,1,1))
        
            # Open menu
            press_menu_button(driver)
        
            # Clear cars
            clear_cars(driver)
        
            # Click off menu
            click_off(y_coords)
            time.sleep(0.1)
        
            # If not last group:
            if group != groups[-1]:
        
                # Open car search
                open_car_search(driver)
            
            # Add group to scraped groups
            scraped_groups[rarity].append(group)

            # Save progress
            scraping_progress = {
                'car_times_and_stats_dicts': car_times_and_stats_dicts,
                'car_info_dicts': car_info_dicts,
                'scraped_groups': scraped_groups,
                'completed_rarities': completed_rarities
            }
            with open('scraping_progress.pkl', 'wb') as f:
                pickle.dump(scraping_progress, f)
            
            # Update progress bar
            pbar.update(1)

        # Close progress bar
        pbar.close()
    
        # Add rarity to completed rarities
        completed_rarities.append(rarity)

        # Save progress
        scraping_progress = {
            'car_times_and_stats_dicts': car_times_and_stats_dicts,
            'car_info_dicts': car_info_dicts,
            'scraped_groups': scraped_groups,
            'completed_rarities': completed_rarities
        }
        with open('scraping_progress.pkl', 'wb') as f:
            pickle.dump(scraping_progress, f)

        first_scrape_rarity = False

    return scraping_progress
#endregion

#   PPPPPP   RRRRRR   EEEEEEE          PPPPPP   RRRRRR    OOOOO     CCCC   EEEEEEE  SSSS     SSSS   IIIIII  NN   NN    GGGGG
#   PP   PP  RR   RR  EE               PP   PP  RR   RR  OO   OO   CC  CC  EE      SS  SS   SS  SS    II    NNN  NN   GG
#   PP   PP  RR   RR  EE               PP   PP  RR   RR  OO   OO  CC       EE      SS       SS        II    NNNN NN  GG
#   PP   PP  RR  RRR  EEEEEE  HHHHHHH  PP   PP  RR  RRR  OO   OO  CC       EEEEEE   SSSSS    SSSSS    II    NNNNNNN  GG  GGG
#   PPPPPP   RRRRR    EE               PPPPPP   RRRRR    OO   OO  CC       EE           SS       SS   II    NN NNNN  GG   GG
#   PP       RR RRR   EE               PP       RR RRR   OO   OO   CC  CC  EE      SS   SS  SS   SS   II    NN  NNN   GG  GG
#   PP       RR  RRR  EEEEEEE          PP       RR  RRR   OOOOO     CCCC   EEEEEEE  SSSSS    SSSSS  IIIIII  NN   NN    GGGGG
#
#region
#endregion

#    OOOOO   WW   WW  NN   NN  EEEEEEE  RRRRRR    SSSS    HH   HH  IIIIII  PPPPPP
#   OO   OO  WW   WW  NNN  NN  EE       RR   RR  SS  SS   HH   HH    II    PP   PP
#   OO   OO  WW W WW  NNNN NN  EE       RR   RR  SS       HH   HH    II    PP   PP
#   OO   OO  WWWWWWW  NNNNNNN  EEEEEE   RR  RRR   SSSSS   HHHHHHH    II    PP   PP
#   OO   OO  WWWWWWW  NN NNNN  EE       RRRRR         SS  HH   HH    II    PPPPPP
#   OO   OO  WWW WWW  NN  NNN  EE       RR RRR   SS   SS  HH   HH    II    PP
#    OOOOO   WW   WW  NN   NN  EEEEEEE  RR  RRR   SSSSS   HH   HH  IIIIII  PP
#
#region
#endregion

#     CCCC   HH   HH    AAA    LL       LL       EEEEEEE  NN   NN    GGGGG  EEEEEEE
#    CC  CC  HH   HH   AA AA   LL       LL       EE       NNN  NN   GG      EE
#   CC       HH   HH  AA   AA  LL       LL       EE       NNNN NN  GG       EE
#   CC       HHHHHHH  AA   AA  LL       LL       EEEEEE   NNNNNNN  GG  GGG  EEEEEE
#   CC       HH   HH  AAAAAAA  LL       LL       EE       NN NNNN  GG   GG  EE
#    CC  CC  HH   HH  AA   AA  LL       LL       EE       NN  NNN   GG  GG  EE
#     CCCC   HH   HH  AA   AA  LLLLLLL  LLLLLLL  EEEEEEE  NN   NN    GGGGG  EEEEEEE
#
#region
#endregion

In [6]:
# Load window details and click off coordinates
with open('device_details.pkl', 'rb') as f:
    device_details = pickle.load(f)
window_width = device_details['width']
window_height = device_details['height']
y_coords = device_details['y_coords']

In [7]:
# Set up Firefox with automatic driver management
service = Service(GeckoDriverManager().install())
options = webdriver.FirefoxOptions()
driver = webdriver.Firefox(service=service, options=options)
#driver.set_window_rect(x=0, y=0, width=window_width, height=window_height)
driver.maximize_window()
driver.execute_script("document.body.style.overflowX = 'hidden';")

In [8]:
# Set up action chains
actions = ActionChains(driver)

In [9]:
# Open website
driver.get('https://www.topdrivesrecords.com/')

In [10]:
# Set up page for scraping
initial_menu_setup(driver, y_coords)

In [11]:
time.sleep(1)

In [12]:
test_mode = False


In [13]:
def set_all_tracks(driver: webdriver, y_coords: int, test_mode = False) -> List:
    '''
    Selects every track and variant from the track selection, and gets their names.

    Parameters:
        driver (webdriver): The webdriver object
        y_coords (int): The y coordinate of the dropdown menu
        test_mode (bool): Whether to run in test mode or not (only adds first track)

    Returns:
        List: A list containing all track and variant combinations
    '''
    # Initialise trackset
    trackset = []

    # Open track selection
    add_tracks_button_xpath = '/html/body/div/div[2]/div[1]/div[1]/div[2]/div/div[2]/button'
    add_tracks_button = driver.find_element(By.XPATH, add_tracks_button_xpath)
    click_element(driver, add_tracks_button)
    time.sleep(0.1)

    # Get location of all track divs
    tracks_box_xpath = '/html/body/div/div[5]/div[2]/div/div/div'
    tracks_box = driver.find_element(By.XPATH, tracks_box_xpath)
    indiv_track_divs = tracks_box.find_elements(By.XPATH, './div')

    # Remove searchbar div
    indiv_track_divs = [div for div in indiv_track_divs if div.get_attribute('class') != 'Track_SearchBox']

    # Count number of tracks
    num_tracks = len(indiv_track_divs)

    # Create progress bar
    pbar = tqdm(total=num_tracks, desc='Adding tracks')


    # Loop through all main divs
    for track_div in indiv_track_divs:
        #print(track_div.get_attribute('innerHTML'))
        # Check if dynamic
        div_1_class = track_div.find_element(By.XPATH, './div[1]').get_attribute('class')
        if div_1_class == 'Main_CustomTrackLeftDynamic':
            continue

        # Get track name
        track_name = track_div.find_element(By.XPATH, './div[1]/div').text

        # Check if track has other conditions (rolling start, bumps)
        # Try to find svg element, indicating other conditions
        try:
            i_el_class = track_div.find_element(By.TAG_NAME, 'i').get_attribute('class')
            if 'clearance' in i_el_class:
                track_name += ' (C)'
            if 'roll' in i_el_class:
                track_name += ' (R)'
        # If no i element, there are no other conditions
        except NoSuchElementException:
            pass

        # Get all buttons for track (now divs)
        conditions_container = track_div.find_element(By.XPATH, './div[2]/div')
        button_divs = conditions_container.find_elements(By.XPATH, './div')

        # Iterate through all buttons
        for b_div in button_divs:
            # Click button
            click_element(driver, b_div)

            # Get track condition type
            b_div_class = b_div.get_attribute('class')
            condition = b_div_class.split(' ')[2]

            # Map condition
            condition_str = condition_map[condition]

            # Append track with condition to trackset
            trackset.append((track_name, condition_str))

        # Update progress bar
        pbar.update(1)

        # If test mode, only add first track (all variants)
        if test_mode:
            break

    # Close track selection
    click_off(y_coords)

    # Close progress bar
    pbar.close()

    # Return the trackset
    return trackset



In [14]:
# Set all tracks
trackset_tuples = set_all_tracks(driver, y_coords, test_mode)

Adding tracks:  86%|████████▋ | 140/162 [00:34<00:05,  4.04it/s]


In [15]:
# Format trackset into a list of strings
trackset = [t[0] + ' / ' + t[1] for t in trackset_tuples]

# Scraping Loop

In [22]:
time.sleep(3)

In [23]:
# reset
click_off(y_coords)
press_menu_button(driver)
clear_cars(driver)
click_off(y_coords)

In [24]:
# try to load old data
try:
    old_data = pd.read_csv('scraped_data.csv')
except FileNotFoundError:
    old_data = None

C:\Users\ollie\AppData\Local\Temp\ipykernel_29836\2189387941.py:3: DtypeWarning: Columns (0: Test Bowl / Dry, 1: Test Bowl / Dirt, 2: Test Bowl (R) / Dry, 3: Test Bowl (R) / Snow) have mixed types. Specify dtype option on import or set low_memory=False.
  old_data = pd.read_csv('scraped_data.csv')


In [25]:
scrape_new = False

In [26]:
# Try to load progress from file
try:
    if scrape_new == True:
        print('Scraping new data, not loading progress.')
        raise FileNotFoundError
    with open('scraping_progress.pkl', 'rb') as f:
        scraping_progress = pickle.load(f)
    print('Loaded progress from file.')
except FileNotFoundError:
    if not scrape_new:
        print('No progress file found. Initialising new dict.')
    scraping_progress = {
        'car_times_and_stats_dicts': [],
        'car_info_dicts': [],
        'scraped_groups': {r: [] for r in ['S', 'A', 'B', 'C', 'D', 'E', 'F']},
        'completed_rarities': []
    }

Loaded progress from file.


In [27]:
scraping_progress = scrape_main(
    driver, 
    trackset, 
    scraping_progress, 
    y_coords,
    scrape_tags=['Year of the Horsepower', 'Touma\'s Collection 2', 'Crown Pursuit']
)

Rarity S already scraped.
Rarity A already scraped.
Rarity B already scraped.
Rarity C already scraped.
Rarity D already scraped.
Scraping rarity E. 6 groups already scraped.
Adding tags
  Touma's Collection 2
  Year of the Horsepower
  Crown Pursuit


Adding E cars: 100%|██████████| 8/8 [00:26<00:00, 13.40s/it]


Scraping rarity F. No groups already scraped.
  Touma's Collection 2
  Year of the Horsepower
  Crown Pursuit


Adding F cars: 100%|██████████| 8/8 [01:39<00:00, 12.40s/it]


In [28]:
car_times_and_stats_dicts = scraping_progress['car_times_and_stats_dicts']
car_info_dicts = scraping_progress['car_info_dicts']

In [29]:
for k, v in scraping_progress.items():
    print(f'{k}: {len(v)}')

car_times_and_stats_dicts: 1435
car_info_dicts: 436
scraped_groups: 7
completed_rarities: 7


In [30]:
len(car_times_and_stats_dicts) / len(car_info_dicts)

3.291284403669725

In [31]:
# Get all unique rq, make_model, and year combinations from car_times_and_stats_dicts
unique_from_times = set()
for car_times_and_stats_dict in car_times_and_stats_dicts:
    rq = car_times_and_stats_dict['rq']
    make_model = car_times_and_stats_dict['make_model']
    year = car_times_and_stats_dict['year']
    unique_from_times.add((rq, make_model, year))

unique_from_infos = set()
for car_info_dict in car_info_dicts:
    rq = car_info_dict['rq']
    make_model = car_info_dict['make_model']
    year = car_info_dict['year']
    unique_from_infos.add((rq, make_model, year))

# Iterate through all car info dicts and check if they are in times_rqs_makes
for unique_from_info in unique_from_infos:
    if unique_from_info not in unique_from_times:
        print(f'{unique_from_info} not in unique_from_times')

# Iterate through all times_rqs_makes and check if they are in car info dicts
for unique_from_time in unique_from_times:
    if unique_from_time not in unique_from_infos:
        print(f'{unique_from_time} not in unique_from_infos')

In [32]:
# Join dicts
# Iterate through all car times and stats dicts
for car_times_and_stats_dict in car_times_and_stats_dicts:
    # Get rq, make_model, and year
    rq = car_times_and_stats_dict['rq']
    make_model = car_times_and_stats_dict['make_model']
    year = car_times_and_stats_dict['year']

    # Find car info dict with same rq and make_model
    car_info_dict = next((car_info_dict for car_info_dict in car_info_dicts if car_info_dict['rq'] == rq and car_info_dict['make_model'] == make_model and car_info_dict['year'] == year), None)
    # If not found, print error
    if car_info_dict is None:
        print(f'Error: {rq} {make_model} not found in car info dicts')
        continue

    # Add car info dict to car times and stats dict
    car_times_and_stats_dict.update(car_info_dict)

In [33]:
# convert to dataframe
scraped_data_df = pd.DataFrame(car_times_and_stats_dicts)

In [34]:
# convert new columns to same type as old columns
scraped_data_df['rq'] = scraped_data_df['rq'].astype(int)
scraped_data_df['year'] = scraped_data_df['year'].astype(int)

In [35]:
# Add unique key col to data
merge_keys = ['rq', 'make_model', 'year', 'engine_up', 'weight_up', 'chassis_up', 'rid']
if old_data is not None:
    old_data['merge_key'] = old_data.apply(lambda x: '/'.join([str(x[k]) for k in merge_keys]), axis=1)
scraped_data_df['merge_key'] = scraped_data_df.apply(lambda x: '/'.join([str(x[k]) for k in merge_keys]), axis=1)

In [36]:
if old_data is not None:
    old_data.set_index('merge_key', inplace=True)
scraped_data_df.set_index('merge_key', inplace=True)

In [37]:
# Set all columns to string type and add column of nans to old data if a new col
for col in scraped_data_df.columns:
    scraped_data_df[col] = scraped_data_df[col].astype(str)
    if old_data is not None:    
        if col not in old_data.columns:
            old_data[col] = 'nan'
        old_data[col] = old_data[col].astype(str)

In [38]:
scraped_data_df.sort_index(inplace=True)

In [39]:
# remove cols from old data that are not in new data
'''
keep_cols = []
for col in old_data.columns:
    if col in scraped_data_df.columns:
        keep_cols.append(col)

old_data = old_data[scraped_data_df.columns]
'''

'\nkeep_cols = []\nfor col in old_data.columns:\n    if col in scraped_data_df.columns:\n        keep_cols.append(col)\n\nold_data = old_data[scraped_data_df.columns]\n'

In [40]:
# Check all cols are the same
old_cols = old_data.columns
new_cols = scraped_data_df.columns
if (old_cols == new_cols).all():
    print('All cols same')
else:
    print('Not same cols')
    for i, col in enumerate(old_cols):
        print(i, col, '///', new_cols[i])
    raise ValueError

All cols same


In [41]:
# Concat new data with old data
combined_data = scraped_data_df.combine_first(old_data)

# Sort by index alphabetically
combined_data.sort_index(inplace=True)

# Reset index
combined_data.reset_index(drop=True, inplace=True)

In [42]:
combined_data.head()

,0-100-0mph / Dry,0-100-0mph / Wet,0-100-0mph / Dirt,0-100mph / Dry,0-100mph / Wet,0-100mph / Dirt,0-100mph / Gravel,0-120mph / Dry,0-120mph / Gravel,0-124mph / Dry,...,abs,tcs,clearance,mra,weight,fuel,seats,engine_pos,body,brakes
0,01:05:24,01:34:77,NaN,00:59:69,01:25:58,NaN,NaN,NaN,NaN,NaN,...,No,No,Low,26.3,535.0,Petrol,2.0,Rear,Coupe,C
1,00:55:91,01:11:25,DNF,00:50:28,01:01:83,DNF,DNF,DNF,DNF,DNF,...,No,No,Low,26.3,535.0,Petrol,2.0,Rear,Coupe,C
2,00:54:34,01:09:36,NaN,00:48:71,00:59:96,NaN,NaN,DNF,NaN,DNF,...,No,No,Low,26.3,535.0,Petrol,2.0,Rear,Coupe,C
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,Low,26.36,820.0,Petrol,2.0,Front,Roadster,C
4,00:41:41,00:49:11,DNF,00:35:73,00:39:56,DNF,00:49:78,DNF,DNF,DNF,...,No,No,Low,26.36,820.0,Petrol,2.0,Front,Roadster,C


In [43]:
combined_data.shape

(20734, 373)

In [44]:
# save to csv
combined_data.to_csv('scraped_data.csv', index=False)